# Analysis of 2016 election results recorded by counties

Each row of the data set is a U.S. county. 
Use the response **two-party Democratic vote share**

$$
Y \;=\; \frac{\texttt{DEM}}{\texttt{DEM} + \texttt{GOP}} \in [0,1],
$$

i.e. the fraction of the votes cast for the two major parties that went to the Democratic candidate, to characterize the distribution of county votes between the `DEM` and `GOP` 
parties. 

Use the following seven predictors:

| Predictor | Meaning (county level) |
|---|---|
| `MedianIncome` | median household income, dollars |
| `MigraRate` | net migration rate (per 1,000 residents) |
| `BirthRate` | births per 1,000 residents |
| `DeathRate` | deaths per 1,000 residents |
| `BachelorRate` | % of adults with a bachelor's degree or higher |
| `UnemploymentRate` | unemployment rate, % |
| `TotalVotes` | total number of votes, `DEM` + `GOP` (a measure of county size) |


## Load and inspect the data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

raw = pd.read_csv("A2016.csv")
print(raw.shape)
raw.head()

(3234, 10)


,FIPS,County,DEM,GOP,MedianIncome,MigraRate,BirthRate,DeathRate,BachelorRate,UnemploymentRate
0,1001,"Autauga County, AL",5908.0,18110.0,54487.0,5.3,12.2,10.0,27.7,5.1
1,1003,"Baldwin County, AL",18409.0,72780.0,56460.0,21.5,11.2,9.8,31.3,5.3
2,1005,"Barbour County, AL",4848.0,5431.0,32884.0,-18.2,10.6,10.8,12.2,8.3
3,1007,"Bibb County, AL",1874.0,6733.0,43079.0,-0.8,12.1,10.8,11.5,6.4
4,1009,"Blount County, AL",2150.0,22808.0,47213.0,-1.0,12.2,11.2,12.6,5.4


In [2]:
print("Missing values per column:")
print(raw.isna().sum())

Missing values per column:
FIPS                  0
County                1
DEM                 122
GOP                 122
MedianIncome         95
MigraRate            94
BirthRate            94
DeathRate            94
BachelorRate         16
UnemploymentRate     17
dtype: int64


## The response and a single common sample

Rows with no `DEM`/`GOP` counts cannot have a response. Most of them are not counties that voted in the 2016
presidential election in the usual way: Puerto Rico and the other territories (which do not vote for president),
and Alaska, which reports results by state-house district rather than by borough. A few more rows are missing a
predictor.

**Use the same complete-case sample for every model.** This matters for the comparison in Section 7: if the
simple regression of $Y$ on `BachelorRate` used 3,112 counties and the multiple regression used 3,111, a change in a
coefficient could be partly a change in *which counties* were fit rather than a change in *which predictors* were
in the model.


**The new predictor `TotalVotes`.** The data contain only the Democratic and Republican counts, so the total number
of votes is computed as `DEM + GOP` (third-party votes are not available). It is essentially a measure of how
populous the county is. Note that it is built from the same two columns as $Y$, but $Y$ is a *ratio* and
`TotalVotes` a *sum*, so neither determines the other.

In [3]:
PREDICTORS = ["MedianIncome", "MigraRate", "BirthRate", "DeathRate", "BachelorRate", "UnemploymentRate",
              "TotalVotes"]

# new predictor: total two-party votes cast in the county
raw["TotalVotes"] = raw["DEM"] + raw["GOP"]

no_votes = raw[raw["DEM"].isna() | raw["GOP"].isna()]
state_abbr = no_votes["County"].str[-2:]
print(f"rows without vote counts: {len(no_votes)}")
print("  by state/territory:", state_abbr.value_counts().to_dict())

df = raw.dropna(subset=["DEM", "GOP"] + PREDICTORS).copy()
df["DemShare"] = df["DEM"] / (df["DEM"] + df["GOP"])
df["state"] = df["FIPS"] // 1000            # first two FIPS digits = state code (used in Section 9)

n = len(df)
print(f"\ncomplete-case sample: n = {n} counties in {df['state'].nunique()} states (incl. DC)")
df[["County", "DEM", "GOP", "DemShare"] + PREDICTORS].head()

rows without vote counts: 122
  by state/territory: {'PR': 78, 'AK': 29, 'AS': 5, 'MP': 4, 'VI': 3, 'HI': 1, 'VA': 1, 'GU': 1}

complete-case sample: n = 3111 counties in 50 states (incl. DC)


,County,DEM,GOP,DemShare,MedianIncome,MigraRate,BirthRate,DeathRate,BachelorRate,UnemploymentRate,TotalVotes
0,"Autauga County, AL",5908.0,18110.0,0.245982,54487.0,5.3,12.2,10.0,27.7,5.1,24018.0
1,"Baldwin County, AL",18409.0,72780.0,0.201877,56460.0,21.5,11.2,9.8,31.3,5.3,91189.0
2,"Barbour County, AL",4848.0,5431.0,0.471641,32884.0,-18.2,10.6,10.8,12.2,8.3,10279.0
3,"Bibb County, AL",1874.0,6733.0,0.217730,43079.0,-0.8,12.1,10.8,11.5,6.4,8607.0
4,"Blount County, AL",2150.0,22808.0,0.086145,47213.0,-1.0,12.2,11.2,12.6,5.4,24958.0
